### Step 1: Import Libraries and load the environment variables

In [39]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display
from pprint import pprint
import gradio as gr
import json

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API key is missing")


### Step 2: Set up Pushover

In [3]:
#Step 2a -> Set up account in your browser
#Step 2b -> Set up the app on your iPhone / Android phone, log into the same account
#Step 2c -> In the browser create an "Application/API Token"
#Step 2d -> Copy uor User Key and API Token into the .env file,
#like this but without the hashtag symbols and with your own keys:
#PUSHOVER_USER=xxxxxxxxxxxx
#PUSHOVER_TOKEN=yyyyyyyyyyyyyy

#Save changes to the .env file

#Also, run the test to manually send a notification from Pushover in your browser to pushover on your phone.


In [40]:
load_dotenv() #rerun to load the changes you made (make sure you saved them)

True

In [41]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

In [18]:
#Use this (in private!) to test that your keys have been loaded.
#print(pushover_user)
#print(pushover_token)

In [42]:
#Test Pushover
import requests

def send_notification(message: str):
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [20]:
#send_notification("Hello to myself, from this amazing AI Engineering training!")

### Step 3: Describe Pushover as an LLM tool

In [43]:
send_notification_function = {
    "name": "send_notification",
    "description": "Sends a push notification to the user's phone via Pushover. Use this to alert the user about important events, completed tasks, or time-sensitive information.",
    "parameters": {
        "type": "object",
        "properties": {
            "message": {
                "type": "string",
                "description": "The notification message to send to the user's device."
            }
        },
        "required": ["message"]
    }
}

### Step 4: Add Pushover to the list of tools for the LLM

In [44]:
tools = [{"type": "function", "function": send_notification_function}]

### Step 2b & 3b & 4b: Create new function, describe it, add it to the list of tools

In [64]:
import random

#Simulates rolling a single six-sided die
def dice_roll():
    result = random.randint(1,6)
    return result

#Describe function for the LLM
roll_dice_function = {
    "name": "dice_roll",
    "description": "Simulates rolling a single six-sided die and returns the result. Use this when the user wants to roll a die for games, decisions, or random number generation.",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": []
    }
}

#Add function to lit of tools of LLM
tools.append({"type": "function", "function": roll_dice_function})

### Step 5: Calling the tool from an LLM

In [58]:
#comenting because we improved the def in the line below
#def handle_tool_call(tool_calls):
    #....
#    tool_call = tool_calls[0] #Assuming just one tool call
#    args = json.loads(tool_call.function.arguments)

    #Actually send the notification, i.e. call the tool
#    send_notification(args["message"])
    
#    tool_call_result = {
#        "role": "tool",
#        "content": f"Notification sent: {args['message']}",
#        "tool_call_id": tool_call.id,
#    }
    #return what to add to our "context" (about tool call results), a dictionary
#    return tool_call_result

In [65]:
def handle_tool_call(tool_calls):
    tool_results = []

    for tool_call in tool_calls:
        function_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        #print(f"Calling function {function_name}") #For future debugging!

        #Route to the appropriate function based on function_name
        if function_name == "send_notification":
            send_notification(args["message"])
            content = f"Notification sent: {args['message']}"
        elif function_name == "dice_roll":
            content = f"Rolled: {dice_roll()}"
        #elif function_name == "insert_function_name_3":
            # content = insert_function_name3(args["message"])
        #....
        else:
            content = f"Unknown function: {function_name}"

        tool_call_result = {
            "role": "tool",
            "content": content,
            "tool_call_id": tool_call.id,
        }

        tool_results.append(tool_call_result)
    
    return tool_results

In [ ]:
client = OpenAI()
messages=[
        {"role": "user", "content": "Please do two things: \
         1) I'd like to roll 2 dice, and \
         2) Send me a notification with the highest of the rolls"}
    ]

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=messages,
    tools=tools
)

message = response.choices[0].message

#Check if model wants to call a tool
while message.tool_calls:
    from pprint import pprint
    pprint(message.tool_calls)
    #...... handle tool call
    tools_result = handle_tool_call(message.tool_calls) #whole list of tool calls on purpose
    #...... add message to "context", i.e. messages
    messages.append(message)
    #...... add info about tool call response to "context", i.e. messages
    messages.extend(tools_result) #changed from append to extend when we switched to multiple tool call handling
    #...... invoke the LLM one more time to get its updated response
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        tools=tools
    )
    message = response.choices[0].message

    #Note: maybe consider adding protection from infinite consecutive tool calling

    #...... print(message.content) #from the new LLM response
print(message.content)

[ChatCompletionMessageFunctionToolCall(id='call_feAfi80TFBUlZdy54Y8DBxi9', function=Function(arguments='{}', name='dice_roll'), type='function'),
 ChatCompletionMessageFunctionToolCall(id='call_rAehv6BYOfu2ZQlTFnY8lym9', function=Function(arguments='{}', name='dice_roll'), type='function')]
[ChatCompletionMessageFunctionToolCall(id='call_w0z4HUMAaUWVPo4UJN0T4dcC', function=Function(arguments='{"message":"The highest roll out of your two dice is 3."}', name='send_notification'), type='function')]
I rolled two dice for you. The results were 3 and 1. I have sent you a notification with the highest roll, which is 3.
